# 04 CNN Training

Stage 7 prepares PyTorch-ready tensors for deep learning. This notebook starts with shape, leakage, and preprocessing checks for a single-epoch 1D CNN dataset, then includes a tiny overfit smoke test for the first CNN training loop.

In [ ]:
from pathlib import Path

import torch
from torch import nn

from src.data import (
    DEFAULT_EPOCH_INDEX_PATH,
    DEFAULT_PREPROCESSING_METADATA_PATH,
    DEFAULT_RAW_DATA_DIR,
    DreamtContextDataset,
    DreamtEpochDataset,
    DreamtSequenceDataset,
    check_epoch_split_leakage,
    create_dataloaders,
    fit_normalization_stats,
    load_preprocessing_metadata,
    save_preprocessing_metadata,
)

CHANNELS = ["BVP", "ACC_X", "ACC_Y", "ACC_Z", "TEMP", "EDA", "HR", "IBI"]
BATCH_SIZE = 16
DEBUG_PARTICIPANTS = 3

raw_dir = DEFAULT_RAW_DATA_DIR
epoch_index_path = DEFAULT_EPOCH_INDEX_PATH
metadata_path = DEFAULT_PREPROCESSING_METADATA_PATH

artifacts_available = raw_dir.exists() and epoch_index_path.exists()
artifacts_available


## Build Single-Epoch Datasets

In [ ]:
if artifacts_available:
    train_unscaled = DreamtEpochDataset(
        raw_dir=raw_dir,
        epoch_index=epoch_index_path,
        split="train",
        channels=CHANNELS,
        max_participants=DEBUG_PARTICIPANTS,
    )
    stats = fit_normalization_stats(train_unscaled)
    save_preprocessing_metadata(stats, metadata_path)
else:
    print("Skipping dataset construction because local raw files or epoch_index.csv are absent.")


In [ ]:
if artifacts_available:
    stats = load_preprocessing_metadata(metadata_path)
    train_ds = DreamtEpochDataset(raw_dir, epoch_index_path, split="train", channels=CHANNELS, preprocessing_stats=stats, max_participants=DEBUG_PARTICIPANTS)
    val_ds = DreamtEpochDataset(raw_dir, epoch_index_path, split="validation", channels=CHANNELS, preprocessing_stats=stats, max_participants=DEBUG_PARTICIPANTS)
    test_ds = DreamtEpochDataset(raw_dir, epoch_index_path, split="test", channels=CHANNELS, preprocessing_stats=stats, max_participants=DEBUG_PARTICIPANTS)
    check_epoch_split_leakage(train_ds.epoch_index)
    check_epoch_split_leakage(val_ds.epoch_index)
    check_epoch_split_leakage(test_ds.epoch_index)
    loaders = create_dataloaders(train_ds, val_ds, test_ds, batch_size=BATCH_SIZE)
    x_batch, y_batch = next(iter(loaders["train"]))
    print("train batch:", tuple(x_batch.shape), x_batch.dtype, tuple(y_batch.shape), y_batch.dtype)
    print("participants:", {"train": len(train_ds.participants), "validation": len(val_ds.participants), "test": len(test_ds.participants)})
    print("metadata channels:", stats["channels"])


## Temporal Context And Sequence Shape Checks

In [ ]:
if artifacts_available:
    context_ds = DreamtContextDataset(raw_dir, epoch_index_path, split="train", channels=CHANNELS, preprocessing_stats=stats, context_radius=2, max_participants=DEBUG_PARTICIPANTS)
    sequence_ds = DreamtSequenceDataset(raw_dir, epoch_index_path, split="train", channels=CHANNELS, preprocessing_stats=stats, sequence_length=5, label_mode="many_to_one", target_position="center", max_participants=DEBUG_PARTICIPANTS)
    if len(context_ds):
        x_context, y_context = context_ds[0]
        print("context item:", tuple(x_context.shape), y_context.item())
    if len(sequence_ds):
        x_sequence, y_sequence = sequence_ds[0]
        print("sequence item:", tuple(x_sequence.shape), y_sequence.item())


## Tiny Overfit Smoke Test

In [ ]:
class TinySleepStageCNN(nn.Module):
    def __init__(self, channels, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(channels, 16, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(16, n_classes),
        )

    def forward(self, x):
        return self.net(x)


def tiny_overfit_test(dataset, steps=25, batch_size=8, lr=1e-2):
    subset_size = min(len(dataset), batch_size)
    if subset_size < 2:
        print("Need at least two epochs for the tiny overfit smoke test.")
        return []
    loader = torch.utils.data.DataLoader(torch.utils.data.Subset(dataset, range(subset_size)), batch_size=subset_size, shuffle=True)
    model = TinySleepStageCNN(channels=len(dataset.channels)).double()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    losses = []
    x, y = next(iter(loader))
    for _ in range(steps):
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        losses.append(float(loss.item()))
    print("loss first/last:", round(losses[0], 4), round(losses[-1], 4))
    return losses


if artifacts_available:
    losses = tiny_overfit_test(train_ds)
